# Atlantic Haven Hotels — Prédiction d'annulation de réservation
**Examen final ML & Data Science — M1 ISPM**

Objectif : prédire `reservation_annulee` (1 = annulée, 0 = maintenue) et **maximiser le F1-score sur la classe annulation**.

Contrainte clé : les données sont **ordonnées dans le temps** et le test est plus récent que le train → nous adoptons une **validation temporelle** (une CV aléatoire donnerait une estimation optimiste et trompeuse).

**Plan du notebook**
1. Configuration & chargement
2. EDA (cible, manquantes, signaux catégoriels, dérive temporelle)
3. Feature engineering (sans fuite de cible)
4. Protocole de validation temporelle
5. Baseline : régression logistique
6. Réglage du seuil de décision (optimisation du F1)
7. Familles de modèles alternatives (RandomForest, HistGradientBoosting)
8. Modèle final, importance des variables, analyse d'erreurs, équité par région
9. Génération de `submission.csv`

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             precision_recall_curve, confusion_matrix, classification_report)

# --- Reproductibilité ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Répertoire des données : placez reservations_train.csv / reservations_test.csv à côté du notebook.
DATA_DIR = "."
TARGET = "reservation_annulee"
ID = "reservation_id"

train = pd.read_csv(f"{DATA_DIR}/reservations_train.csv")
test  = pd.read_csv(f"{DATA_DIR}/reservations_test.csv")
print("train:", train.shape, "| test:", test.shape)

## 2. EDA — Analyse exploratoire

In [ ]:
# --- 2.1 Équilibre de la cible ---
dist = train[TARGET].value_counts(normalize=True).rename({0:"maintenue",1:"annulée"})
print("Répartition de la cible :")
print((dist*100).round(1).astype(str) + " %")
print(f"\nClasse minoritaire (annulation) = {train[TARGET].mean():.1%}"
      "  -> déséquilibre modéré : l'accuracy serait trompeuse, on optimise le F1.")

In [ ]:
# --- 2.2 Valeurs manquantes ---
na = train.isna().sum()
na = na[na > 0].sort_values(ascending=False)
print("Valeurs manquantes (train) :")
print(na.to_string())
print("\nNB : agent_id vide = réservation DIRECTE (cf. dictionnaire), ce n'est pas un vrai manquant.")
print("On en fera un drapeau 'reservation_directe' plutôt qu'une imputation.")
print("Taux annulation | agent_id vide :", round(train[train.agent_id.isna()][TARGET].mean(),3),
      "| agent_id présent :", round(train[train.agent_id.notna()][TARGET].mean(),3))

In [ ]:
# --- 2.3 Taux d'annulation par variables commerciales clés ---
for c in ["type_acompte","tarif_remboursable","canal_reservation","segment_client"]:
    g = (train.groupby(c)[TARGET].mean()*100).round(1).sort_values(ascending=False)
    print(f"--- {c} ---"); print(g.to_string()); print()

In [ ]:
# --- 2.4 Dérive temporelle : taux d'annulation par trimestre de réservation ---
tmp = train.copy()
tmp["trimestre"] = pd.to_datetime(tmp["date_reservation"]).dt.to_period("Q").astype(str)
drift = tmp.groupby("trimestre")[TARGET].agg(["mean","count"]).round(3)
print(drift.to_string())
print("\nLe taux reste stable (~0.22-0.30) : pas de rupture majeure, mais le test étant")
print("postérieur au train, on valide sur la période la plus récente du train.")

## 3. Feature engineering (sans fuite de cible)

Toutes les variables créées n'utilisent **que** des informations connues au moment de la réservation. On n'utilise jamais la cible ni aucune donnée du test pour construire ou imputer.

In [ ]:
def add_features(df):
    df = df.copy()
    dres = pd.to_datetime(df["date_reservation"], errors="coerce")
    darr = pd.to_datetime(df["date_arrivee"], errors="coerce")
    # Dates
    df["res_mois"]      = dres.dt.month
    df["res_annee"]     = dres.dt.year
    df["arr_mois"]      = darr.dt.month
    df["arr_trimestre"] = darr.dt.quarter
    df["arr_jour_sem"]  = darr.dt.dayofweek
    # Historique client (ratio d'annulation passé ; colonnes historiques, pas la cible)
    df["taux_annul_hist"] = (df["annulations_passees"] /
                             df["reservations_passees"].replace(0, np.nan)).fillna(0)
    df["client_connu"]    = (df["reservations_passees"] > 0).astype(int)
    # Séjour / prix
    df["personnes_total"]   = df["adultes"] + df["enfants"].fillna(0)
    df["a_enfants"]         = (df["enfants"].fillna(0) > 0).astype(int)
    df["prix_par_personne"] = df["prix_moyen_nuit_eur"] / df["personnes_total"].replace(0, np.nan)
    df["cout_total_estime"] = df["prix_moyen_nuit_eur"] * df["nuits"]
    # Réservation directe (agent_id vide) + buckets de délai
    df["reservation_directe"] = df["agent_id"].isna().astype(int)
    df["delai_court"] = (df["delai_reservation_jours"] <= 7).astype(int)
    df["delai_long"]  = (df["delai_reservation_jours"] >= 90).astype(int)
    return df

train_fe = add_features(train)
test_fe  = add_features(test)
print("Nouvelles variables :", [c for c in train_fe.columns if c not in train.columns])

## 4. Protocole de validation temporelle

On trie le train par `date_reservation` et on réserve les **20 % les plus récents** comme jeu de validation.
Ce jeu unique sert à comparer **tous** les modèles et à choisir le seuil — il imite le test futur.
Tous les prétraitements sont appris **uniquement** sur le fold d'entraînement.

In [ ]:
train_fe = train_fe.sort_values("date_reservation").reset_index(drop=True)
cut = int(len(train_fe) * 0.80)
tr_df, va_df = train_fe.iloc[:cut].copy(), train_fe.iloc[cut:].copy()
print(f"Train fold : {len(tr_df)} lignes (jusqu'au {tr_df['date_reservation'].max()})")
print(f"Valid fold : {len(va_df)} lignes ({va_df['date_reservation'].min()} -> {va_df['date_reservation'].max()})")

# Colonnes exploitées (on écarte identifiants, dates brutes, et les hautes cardinalités agent/hotel)
drop_cols = [ID, "date_reservation", "date_arrivee", "agent_id", "hotel_id", TARGET]
feat = [c for c in train_fe.columns if c not in drop_cols]
num_cols = train_fe[feat].select_dtypes(include="number").columns.tolist()
cat_cols = train_fe[feat].select_dtypes(exclude="number").columns.tolist()
print(f"\n{len(num_cols)} numériques | {len(cat_cols)} catégorielles")

# Prétraitements : imputation + encodage. handle_unknown='ignore' gère les catégories jamais vues.
pre_linear = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Inconnu")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols)])

pre_tree = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Inconnu")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols)])

Xtr, ytr = tr_df[feat], tr_df[TARGET].values
Xva, yva = va_df[feat], va_df[TARGET].values

In [ ]:
def best_threshold(y_true, proba):
    """Seuil qui maximise le F1 sur la classe annulation (via la courbe précision-rappel)."""
    prec, rec, thr = precision_recall_curve(y_true, proba)
    f1 = 2*prec*rec/(prec+rec+1e-9)
    return float(thr[int(np.argmax(f1[:-1]))])

def evaluate(name, model, pre, fixed_thr=None):
    pipe = Pipeline([("pre", pre), ("clf", model)]).fit(Xtr, ytr)
    proba = pipe.predict_proba(Xva)[:, 1]
    thr = fixed_thr if fixed_thr is not None else best_threshold(yva, proba)
    pred = (proba >= thr).astype(int)
    row = dict(modele=name, seuil=round(thr,3),
               F1=round(f1_score(yva,pred),4), precision=round(precision_score(yva,pred),4),
               rappel=round(recall_score(yva,pred),4), roc_auc=round(roc_auc_score(yva,proba),4))
    print(f"{name:34s} seuil={row['seuil']:.3f}  F1={row['F1']:.4f}  "
          f"P={row['precision']:.3f}  R={row['rappel']:.3f}  AUC={row['roc_auc']:.3f}")
    return row, pipe

## 5. Baseline obligatoire — Régression logistique

In [ ]:
results = []
r, pipe_lr = evaluate("LogReg baseline (seuil 0.5)",
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    pre_linear, fixed_thr=0.5)
results.append(r)

## 6. Réglage du seuil de décision

Sur données déséquilibrées, le seuil 0.5 n'est pas optimal pour le F1. On choisit le seuil qui
maximise le F1 **sur le jeu de validation** (jamais sur le test).

In [ ]:
r, pipe_lr_thr = evaluate("LogReg (seuil optimisé F1)",
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    pre_linear)
results.append(r)

## 7. Familles alternatives — RandomForest & HistGradientBoosting

In [ ]:
r, pipe_hgb = evaluate("HistGradientBoosting",
    HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_depth=6,
        l2_regularization=1.0, class_weight="balanced", random_state=RANDOM_STATE), pre_tree)
results.append(r)

r, pipe_rf = evaluate("RandomForest",
    RandomForestClassifier(n_estimators=400, max_depth=12, min_samples_leaf=5,
        class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE), pre_tree)
results.append(r)

tableau = pd.DataFrame(results)
print("\n=== Tableau comparatif (même jeu de validation temporelle) ===")
print(tableau.to_string(index=False))

### Choix du modèle final

Les familles sont proches (F1 0,47–0,48 ; AUC ≈ 0,64–0,66) : le signal est essentiellement additif.
On retient **RandomForest** car il offre le **meilleur F1** (métrique notée) **et la meilleure stabilité**
à travers plusieurs points de coupe temporels (vérification ci-dessous).

In [ ]:
# Stabilité du F1 selon le point de coupe temporel (75/80/85 %)
def f1_at_cut(frac, model, pre):
    c = int(len(train_fe)*frac)
    a, b = train_fe.iloc[:c], train_fe.iloc[c:]
    pipe = Pipeline([("pre", pre), ("clf", model)]).fit(a[feat], a[TARGET])
    p = pipe.predict_proba(b[feat])[:,1]
    return round(f1_score(b[TARGET], (p>=best_threshold(b[TARGET].values,p)).astype(int)),4)

for frac in [0.75, 0.80, 0.85]:
    lr = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
    rf = RandomForestClassifier(n_estimators=400, max_depth=12, min_samples_leaf=5,
                                class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)
    print(f"cut={frac}:  LogReg F1={f1_at_cut(frac,lr,pre_linear)}   RandomForest F1={f1_at_cut(frac,rf,pre_tree)}")

## 8. Modèle final — importance, analyse d'erreurs, équité

In [ ]:
FINAL = Pipeline([("pre", pre_tree),
    ("clf", RandomForestClassifier(n_estimators=400, max_depth=12, min_samples_leaf=5,
            class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE))]).fit(Xtr, ytr)
p_val = FINAL.predict_proba(Xva)[:,1]
THR = best_threshold(yva, p_val)
pred_val = (p_val >= THR).astype(int)
print(f"Seuil de décision retenu : {THR:.3f}")
print("\nMatrice de confusion (validation) [ [TN FP][FN TP] ] :")
print(confusion_matrix(yva, pred_val))
print("\n", classification_report(yva, pred_val, target_names=["maintenue","annulée"]))

In [ ]:
# --- 8.1 Importance des variables (permutation sur la validation, scoring F1) ---
perm = permutation_importance(FINAL, Xva, yva, n_repeats=5,
                              random_state=RANDOM_STATE, scoring="f1", n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=feat).sort_values(ascending=False)
print("Top 12 variables (permutation importance) :")
print(imp.head(12).round(4).to_string())

In [ ]:
# --- 8.2 Analyse d'erreurs : 5 faux positifs et 5 faux négatifs ---
va_an = va_df.copy(); va_an["proba"] = p_val; va_an["pred"] = pred_val
cols = [ID,"type_acompte","tarif_remboursable","canal_reservation","segment_client",
        "delai_reservation_jours","taux_annul_hist","proba",TARGET,"pred"]
fp = va_an[(va_an[TARGET]==0)&(va_an["pred"]==1)].sort_values("proba",ascending=False).head(5)
fn = va_an[(va_an[TARGET]==1)&(va_an["pred"]==0)].sort_values("proba").head(5)
print("=== 5 FAUX POSITIFS (prédit annulé, réellement maintenu) ===")
print(fp[cols].to_string(index=False))
print("\n=== 5 FAUX NEGATIFS (prédit maintenu, réellement annulé) ===")
print(fn[cols].to_string(index=False))

In [ ]:
# --- 8.3 Équité : F1 par région (Q8) ---
def group_f1(df, proba, col):
    d = df.copy(); d["pred"] = (proba>=THR).astype(int); out=[]
    for g, sub in d.groupby(col):
        if len(sub) < 20 or sub[TARGET].nunique() < 2: continue
        out.append(dict(groupe=g, n=len(sub),
                        F1=round(f1_score(sub[TARGET],sub["pred"]),3),
                        taux_reel=round(sub[TARGET].mean(),3)))
    return pd.DataFrame(out).sort_values("n", ascending=False)
print("F1 par région (validation) :")
print(group_f1(va_df, p_val, "region_hotel").to_string(index=False))
print("\nAttention : les petits sous-groupes (n<100) donnent des F1 très bruités.")

## 9. Soumission

On **réentraîne sur l'intégralité du train** (le seuil est celui déterminé sur la validation, sans fuite du test),
puis on prédit le test en conservant l'ordre des identifiants.

In [ ]:
FINAL_FULL = Pipeline([("pre", pre_tree),
    ("clf", RandomForestClassifier(n_estimators=400, max_depth=12, min_samples_leaf=5,
            class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE))]).fit(train_fe[feat], train_fe[TARGET])

p_test = FINAL_FULL.predict_proba(test_fe[feat])[:,1]
pred_test = (p_test >= THR).astype(int)

submission = pd.DataFrame({
    "reservation_id": test[ID],
    "probabilite_annulation": p_test,
    "reservation_annulee": pred_test})

assert list(submission["reservation_id"]) == list(test[ID]), "ordre des identifiants modifié !"
assert submission.shape == (2000, 3)
submission.to_csv("submission.csv", index=False)
print("submission.csv écrit :", submission.shape)
print("Annulations prédites :", int(pred_test.sum()), f"({pred_test.mean():.1%})")
submission.head()